# Module 5a: Export the apartment sales for Python (Databricks)

**Goal:** one file with every apartment sale from 2021 onwards (outliers excluded), so you can do the statistics for Q2 in Python on your Mac.
**Time:** 15 minutes.

Modules 5 and 6 run in **VS Code**, not Databricks. Real analytics teams split work like this all the time: heavy data prep runs on the platform, and the statistical work runs wherever the analyst is most productive.

## 🧠 Why we're doing this: Parquet instead of CSV

| | CSV | Parquet |
|-|-----|---------|
| Types | Everything is text again; pandas has to guess | Keeps `int`, `float` and `date` exactly |
| Size (this extract) | about 60 MB | about **9 MB** (compressed by column) |
| Read speed | Slow, row by row | Fast; can read only the columns you need |
| Open in Excel? | Yes | No |

So the rule of thumb: **CSV for people, Parquet for programs.** That's why the small summary tables from Module 3 were CSV (Excel will use them) and this extract is Parquet (only Python reads it).

**`.toPandas()` on 580,000 rows?** In Module 3 I said to keep `.toPandas()` for small results. 580k rows × 10 columns is about 50 MB in memory: fine for one machine. The 1M-row fact table with all segments would also fit, but there's no reason to move data the analysis doesn't need. **Filter first, then move.**

In [ ]:
CATALOG, SCHEMA = "workspace", "dubai_re"
T = lambda name: f"{CATALOG}.{SCHEMA}.{name}"
EXPORT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/exports/"

def check_close(label, actual, expected, tolerance=0.01):
    ok = abs(actual - expected) <= tolerance * abs(expected)
    print(f"{'✅' if ok else '❌'} {label}: got {actual:,}, expected about {expected:,} (±{tolerance:.0%})")

✏️ **TODO:** complete the `WHERE` clause so the extract keeps only:
- the `Apartment` segment
- rows that are **not** price outliers
- years **2021 onwards** (the post-COVID cycle; earlier years have too few off-plan vs ready pairs)
- complete months only (`d.is_complete_month`)

In [ ]:
q2 = spark.sql(f"""
    SELECT f.transaction_id, d.month_start, d.year,
           l.community, p.bedroom_band, p.bedroom_sort,
           f.market, f.price_aed, f.area_sqft, f.price_per_sqft
    FROM {T("fact_sales")}   f
    JOIN {T("dim_date")}     d ON f.date_key     = d.date_key
    JOIN {T("dim_location")} l ON f.location_key = l.location_key
    JOIN {T("dim_property")} p ON f.property_key = p.property_key
    WHERE p.segment = 'Apartment'
      AND NOT f.is_price_outlier
      AND d.year >= 2021
      AND d.is_complete_month
""")

n = q2.count()
check_close("Apartment sales in the extract", n, 582_612)

In [ ]:
pdf = q2.toPandas()
pdf.attrs = {}      # remove Spark's hidden run statistics, which can't be saved in Parquet
pdf.to_parquet(EXPORT_PATH + "q2_apartments.parquet", index=False)

print(f"✅ q2_apartments.parquet written ({len(pdf):,} rows)")
print(pdf.dtypes)
display(dbutils.fs.ls(EXPORT_PATH))

## Download it

1. **Catalog → workspace → dubai_re → Volumes → exports → `q2_apartments.parquet` → Download**
2. In your repo, create the folder `data/extracts/` and save the file there.

`.gitignore` already ignores `*.parquet`, so this file stays on your Mac and doesn't go to GitHub. The README explains how to recreate it, which is the right practice for row-level data.

**Commit:** export this notebook to `databricks/05_export_q2_extract.ipynb`, then open `python/05_q2_offplan_vs_ready.ipynb` in VS Code. If it's your first Python notebook in VS Code, set up your environment first with `modules/05_vscode_setup.md`.